In [5]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
   전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */
.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */
.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */
.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */
.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */
.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */
.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}

</style>
"""))

# 1. 데이터 셋

In [6]:
import pandas as pd
url = 'https://raw.githubusercontent.com/4aix/data/refs/heads/master/ch13_apt_fillna_median.csv'
df = pd.read_csv(url)
# df = pd.read_csv('C:/ai/source/01_python/주택도시보증공사_전국 신규 민간아파트 분양가격 동향_20260630.csv', encoding='cp949')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2176 entries, 0 to 2175
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   지역명     2176 non-null   object 
 1   평당분양가격  2176 non-null   float64
 2   연도      2176 non-null   int64  
 3   월       2176 non-null   int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 68.1+ KB


- 지역명2 : 지역명필드를 라벨인코딩하여 추가
- 독립변수(X) : 지역명, 연도, 월
- 종속변수(Y) : 평당분양가격
- 독립변수와 종속변수(reshape)의 스케일 조정
    * 정규화(MinMaxScaler) 작업후 : 지역명2m, 연도m, 월m 컬럼, 평당분양가격m => df_m
    * 표준화(StandardScaler) 작업후 : 지역명2s, 연도s, 월s 컬럼, 평당분양가격s => df_s
    => 지역명, 연도, 월, 지역명2, 지역명2m, 연도m, 월m, 평당분양가격m, 지역명2s, 연도s, 월s컬럼, 평당분양가격s
    
- 데이터프레임.to_numpy(), 데이터프레임.values, np.array(데이터프레임)등을 이용하여 데이터프레임을 넘파이배열로 변환

# 2. 지역명의 라벨인코딩

In [7]:
# sklearn: 머신러닝 모델, 데이터 전처리, 평가 도구 등을 제공하는 핵심 라이브러리 --> sklearn.preprocessing: sklearn 내부의 전처리 전용 모듈
# LabelEncoder: 문자열 라벨을 정수로 변환해주는 인코딩 클래스
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# fit: 데이터를 보고 변환 규칙을 학습 + transform: 학습한 규칙을 이용해서 실제 데이터를 변환
# ┗  fit_transform: 2개
df['지역명2'] = le.fit_transform(df['지역명'])
df = df[['지역명', '지역명2', '평당분양가격', '연도', '월']]
print(df.head())
print(le.classes_)

  지역명  지역명2   평당분양가격    연도   월
0  서울     8  18189.0  2013  12
1  부산     7   8111.0  2013  12
2  대구     5   8080.0  2013  12
3  인천    11  10204.0  2013  12
4  광주     4   6098.0  2013  12
['강원' '경기' '경남' '경북' '광주' '대구' '대전' '부산' '서울' '세종' '울산' '인천' '전남' '전북'
 '제주' '충남' '충북']


# 3. MinMaxScaling
- 변수의 스케일 차이가 학습에 영향을 줄 수 있기때문에, 숫자의 범위를 일정한 범위로 맞추는 것. -> 0부터 1로

In [8]:
from sklearn.preprocessing import MinMaxScaler

X = df[['지역명2', '연도', '월']]
y = df[['평당분양가격']]

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

df[['지역명2s', '연도s', '월s']] = scaler_x.fit_transform(X)
df[['평당분양가격s']]  = scaler_y.fit_transform(y)

df.head()

,지역명,지역명2,평당분양가격,연도,월,지역명2s,연도s,월s,평당분양가격s
0,서울,8,18189.0,2013,12,0.5000,0.0,1.0,0.328198
1,부산,7,8111.0,2013,12,0.4375,0.0,1.0,0.065274
2,대구,5,8080.0,2013,12,0.3125,0.0,1.0,0.064466
3,인천,11,10204.0,2013,12,0.6875,0.0,1.0,0.119878
4,광주,4,6098.0,2013,12,0.2500,0.0,1.0,0.012757


# 4. StandardScaling

# 5. 지역명을 원핫인코딩
- 서울, 경기, ......